# Training a Dense Neural Network for Customer Purchase Prediction

This notebook demonstrates an end-to-end workflow for building a **Dense Neural Network** in PyTorch for binary classification. The goal is to predict whether a customer is likely to make a purchase based on a set of features, allowing us to drive actionable business logic (like triggering a discount popup).

**Key Steps Included:**
1. **Data Preparation:** Generating synthetic customer data, splitting it into train/validation/test sets, and applying standard feature scaling.
2. **Model Building:** Defining a simple feed-forward dense neural network (`CustomerPredictor`) using PyTorch.
3. **Training & Validation:** Training the network using Binary Cross-Entropy loss and the AdamW optimizer, while monitoring validation loss to save the best model checkpoint.
4. **Evaluation:** Plotting learning curves to check for overfitting and evaluating final performance on the test set (Precision, Recall, F1-score, and Confusion Matrix).
5. **Inference:** Passing a new, unseen customer through the model to predict their purchase probability and execute a simulated business decision.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
# Two things in this notebook are drawn at random by PyTorch
# The starting weights and biases inside nn.Linear, when CustomerPredictor() is created.
# The order of the training rows, because DataLoader(..., shuffle=True) shuffles them at each epoch.
torch.manual_seed(42)

# 1. CREATE THE DATASET (Simulating 1,000 customers)
# 3 features
# n_informative=3, all 3 columns actually help separate the classes.
# n_redundant=0, no columns are linear combinations of the informative columns.
X, y = make_classification(n_samples=1000, n_features=3, n_informative=3,
                           n_redundant=0, random_state=42)

print(X)
print(X.shape)
print(y)
print(y.shape)


In [ ]:
# 2. SPLIT THE DATA (Train: 70%, Val: 15%, Test: 15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [ ]:
# 3. SCALE FEATURES — fit on TRAIN ONLY, then transform val/test (no leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(X_train)
print(X_train.shape)
print(type(X_train))

In [ ]:
# 4. CONVERT TO PYTORCH TENSORS
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print(X_train_t)
print(X_train_t.shape)
print(type(X_train_t))

In [ ]:
# 5. DATALOADER — mini-batches of 32
# TensorDataset combines the features and labels into a single dataset object.
train_dataset = TensorDataset(X_train_t, y_train_t)
# DataLoader creates an iterable over the dataset with support for batching, shuffling, and parallel loading.
# DataLoader cuts the training set into mini-batches and gives them to the for loop one at a time.
# It shuffles the 700 rows at the start of every epoch, because shuffle=True. It takes 32 customers at a time, because batch_size=32.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(train_loader)
print(type(train_loader))
print(list(train_loader))

In [ ]:
# 6. DEFINE THE MODEL
# A simple feedforward neural network with one hidden layer. The hidden layer has 8 neurons and uses ReLU activation.
# The output layer has 1 neuron (for binary classification) and outputs raw logits.
class CustomerPredictor(nn.Module):
    def __init__(self):
        # Initialize the parent class (nn.Module) to set up the internal machinery for the model.
        super().__init__()
        # Define a hidden layer that takes 3 input features and outputs 8 features.
        # This layer will learn to extract useful representations from the input data.
        self.hidden = nn.Linear(3, 8)
        # Define the output layer that takes the 8 features from the hidden layer and outputs a single logit.
        self.output = nn.Linear(8, 1)

    # The forward method defines how the input data flows through the model.
    # It applies the hidden layer, then a ReLU activation, and finally the output layer to produce the final logits.
    def forward(self, x):
        # Apply the hidden layer to the input.
        # this will call the forward method of nn.Linear, which computes a linear transformation of the input.
        x = torch.relu(self.hidden(x))
        # Apply the output layer to the activated hidden layer output to get the final logits.
        return self.output(x)

# Instantiate the model
model = CustomerPredictor()

In [ ]:
print(list(model.parameters()))

In [ ]:
# BCE (Binary Cross Entropy): This is the standard mathematical formula used to measure the difference between a binary prediction and the actual truth.
# WithLogits: This is a crucial PyTorch-specific detail. Neural networks output raw numbers called "logits", which can be any positive or negative number.
# To get a probability between 0 and 1, you normally pass logits through a Sigmoid function.
# BCEWithLogitsLoss combines the Sigmoid function and the BCE calculation into one single, highly optimized step.
# It is much more numerically stable than doing them separately.
loss_function = nn.BCEWithLogitsLoss()

# optim.AdamW: This is the specific optimization algorithm being used.
# model.parameters(): You have to explicitly hand the optimizer the model's internal weights and biases.
# lr=1e-3: This is the learning rate, which controls how big of a step the optimizer takes in the direction of the loss gradient.
# weight_decay=1e-4: This is a regularization technique (specifically, L2 regularization).
# It gently penalizes the model for having weights that are too large. This helps prevent overfitting.
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
epochs = 50
train_losses = []
val_losses = []
CHECKPOINT_PATH = "best_customer_predictor.pt"
best_val_loss = float("inf")

print("Starting Training...")

for epoch in range(epochs):
    # Training mode (turn on dropout, batchnorm, etc.)
    # model.train() doesn't train anything. It just flips a flag, model.training, to True or False.
    # Some layers read that flag and behave differently depending on it.
    model.train()
    epoch_train_loss = 0.0

    for batch_X, batch_y in train_loader:
        # batch_X is a tensor with a shape of [32, 3]
        # batch_y is a tensor with a shape of [32, 1]
        # Clear the old gradients (wipe the slate clean) i.e. gradient of previous batches
        optimizer.zero_grad()
        # After passing through your CustomerPredictor's hidden and output layers,
        # the model spits out a tensor with a shape of [32, 1].
        # This is a single column of 32 different raw numbers (logits.
        logits = model(batch_X)
        # applies sigmoid plus BCE, averaged over the 32 rows, giving one number.
        loss = loss_function(logits, batch_y)
        # Compute gradients (backpropagation)
        loss.backward()
        # Update the model's parameters based on the computed gradients
        optimizer.step()
        # Accumulate the training loss for this epoch
        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Evaluation mode (turn off dropout, batchnorm, etc.)
    model.eval()
    # We don't need gradients for validation, so we wrap in torch.no_grad()
    # This is a common pattern to save memory and computations during evaluation.
    # It tells PyTorch that we won't be calling backward() on this block, so it can skip storing intermediate gradients.
    # During any forward pass, PyTorch saves a record of every operation, along with the intermediate values backprop would need later.
    # That record is the grad_fn chain.
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = loss_function(val_logits, y_val_t)
    val_losses.append(val_loss.item())

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        # This extracts the model's state_dict (a simple dictionary containing all the current, perfectly-tuned weights and biases)
        # and saves it to a physical file on your hard drive (best_customer_predictor.pt)
        # We can later load these parameters into a model instance with the same architecture.
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss.item():.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Training Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='orange')
plt.title('Learning Curve: Checking for Overfitting')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_epoch = val_losses.index(min(val_losses)) + 1
print(best_epoch, min(val_losses))

In [ ]:
# Load the best model parameters from the checkpoint file
try:
    # map_location="cpu": This is a safety measure. If the model was trained and saved on a GPU,
    # this forces PyTorch to load it onto the CPU. This prevents errors if you are trying to load
    # the model on a machine that doesn't have a GPU.
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:  # PyTorch < 2.0
    # except TypeError: If the user is running a version of PyTorch older than 2.0,
    # the weights_only argument doesn't exist and will throw a TypeError.
    # The code catches this error and simply runs the load command the old way.
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
# Load the best checkpoint's state_dict into the model. This will update the model's parameters to the best performing ones found during training.
# if you are going to load this in a seperate notebook, you will need to define the model architecture again before loading the state_dict.
# example: model = CustomerPredictor()
model.load_state_dict(ckpt)
print(f"Training Complete! Loaded best checkpoint (val loss {best_val_loss:.4f}).")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


# 1. TEST SET — model already holds best val weights from training cell
# Evaluation mode disables dropout, batch norm updates, etc. — ensures consistent behavior during evaluation/inference
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    # Sigmoid converts raw logits to probabilities between 0 and 1. We then convert to numpy for sklearn metrics.
    # Note: We use .ravel() to flatten the arrays to 1D, which is the expected shape for sklearn metrics.
    test_proba = torch.sigmoid(test_logits).numpy().ravel()
    # Flatten the arrays to 1D
    y_true = y_test_t.numpy().ravel()
    # Convert probabilities to binary predictions using a threshold of 0.5
    y_pred = (test_proba >= 0.5).astype(int)

print(f"y pred: {y_pred}")

# --- THE SIMPLIFIED EVALUATION ---

print("\n--- Test set metrics ---")
# 1. Print Accuracy, Precision, Recall, and F1 all at once
print(classification_report(y_true, y_pred, zero_division=0))

# 2. Print the Confusion Matrix separately
print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# 2. INFERENCE — raw feature values, same scaler as training (fit on train only)
new_customer_raw = [[1.0, 2.0, 3.0]]
new_customer = torch.tensor(scaler.transform(new_customer_raw), dtype=torch.float32)

model.eval()
with torch.no_grad():
    predicted_logit = model(new_customer)
    p_buy = torch.sigmoid(predicted_logit).item()
    print(f"predicted probability: {p_buy}")

# if we assume decision threshold as 0.5

if p_buy > 0.5:
    print("Customer is likely to buy! Action: Trigger a '10% off checkout' coupon popup!")
else:
    print("Customer is unlikely to buy. Action: Do nothing, let them browse.")